# kathareel — Kaggle demo (by raghv)

**Hindi voiceover, English on screen.** Ye notebook poora pipeline chalata hai:
content → Hindi neural voiceover → frame timings → Remotion render → video.

| Step | Kya hota hai |
|---|---|
| 1 | system packages (ffmpeg, Chromium libs) |
| 2 | repo clone |
| 3 | npm install + typecheck |
| 4 | scripts/build_voice.py — Hindi voice + src/timeline.ts |
| 5 | remotion render — 84 sec MP4 |
| 6 | video player + frames + audio preview |

**Notebook settings:** Internet **ON** (warna clone/npm/voice nahi hoga), Accelerator **None** (CPU kaafi hai).

> Voice: hi-IN-MadhurNeural (male). Female ke liye --voice hi-IN-SwaraNeural.
> Internet band ho to --engine piper (offline Hindi voice) use karo.


## 1. System packages
ffmpeg (audio/video) + Chromium ke shared libs jo Remotion ko chahiye.


In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg libnss3 libnspr4 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libasound2 2>/dev/null | tail -2
!ffmpeg -version | head -1
!node -v && npm -v


## 2. Repo clone


In [ ]:
import os, subprocess
REPO = 'https://github.com/amaniimtuniversity-ux/ea_auto.git'
WORK = '/kaggle/working/kathareel'

if not os.path.isdir(f'{WORK}/.git'):
    subprocess.run(['git','clone','--depth','1',REPO,WORK], check=True)
else:
    subprocess.run(['git','-C',WORK,'pull'], check=False)
print(os.listdir(WORK))


## 3. Install + typecheck


In [ ]:
!cd {WORK} && npm install --silent 2>&1 | tail -3
!cd {WORK} && npx tsc --noEmit && echo 'TYPECHECK OK'


## 3b. Python packages (voice ke liye ZAROORI)

edge-tts = natural Hindi neural voice, numpy = audio handling. Ye cell **skip mat karo**, warna
`ModuleNotFoundError: No module named 'edge_tts'` aayega.

Internet nahi hai? Iske bajaye: `!pip install -q piper-tts numpy` aur cell 4 me `--engine piper` lagao.


In [ ]:
!pip install -q edge-tts numpy 2>&1 | tail -3
import edge_tts, numpy
print('edge-tts OK · numpy OK')


## 4. Hindi voiceover + timeline

build_voice.py har `say` line ko synthesize karta hai aur `src/timeline.ts` likhta hai (kis beat ka kaunsa frame). Audio: `public/voice/rag-demo.wav`.

Female: `--voice hi-IN-SwaraNeural` · dheema/tez: `--rate=-8%` · offline: `--engine piper`


In [ ]:
import subprocess, os

# 1) natural neural Hindi voice try karo
r = subprocess.run(['python3', 'scripts/build_voice.py'], cwd=WORK, capture_output=True, text=True)
print(r.stdout[-1200:])

# 2) fail hua to offline voice (piper) se retry
if r.returncode != 0 or not os.path.exists(f'{WORK}/public/voice/rag-demo.wav'):
    print(r.stderr[-600:])
    print('>>> edge voice fail hui - offline piper voice install kar raha hoon...')
    subprocess.run(['pip', 'install', '-q', 'piper-tts', 'numpy'], check=False)
    r = subprocess.run(['python3', 'scripts/build_voice.py', '--engine', 'piper'],
                       cwd=WORK, capture_output=True, text=True)
    print(r.stdout[-1500:])
    if r.returncode != 0:
        print(r.stderr[-1000:])
        raise SystemExit('voice nahi bani - upar ka message dekho')

wav = f'{WORK}/public/voice/rag-demo.wav'
print('\nvoice ready:', wav, round(os.path.getsize(wav) / 1e6, 1), 'MB')


## 5. Timings verify karo


In [ ]:
import re
tl = open(f'{WORK}/src/timeline.ts', encoding='utf-8').read()
total = int(re.search(r'TOTAL_FRAMES = (\d+)', tl).group(1))
print(f'total frames: {total}  =  {total/30:.1f} sec')
for m in re.finditer(r"\{n: (\d+), title: '([^']+)', from: (\d+), to: (\d+)\}", tl):
    n, title, f0, f1 = m.groups()
    print(f'  part {n}: {title:<12} f{f0}-{f1}  ({(int(f1)-int(f0))/30:.1f}s)')
print()
for m in re.finditer(r"\{id: '(\w+)'.*?from: (\d+), to: (\d+), say: '([^']*)'", tl):
    bid, f0, f1, say = m.groups()
    print(f'  {bid:<3} f{f0}-{f1}  {say[:60]}')


## 6. Render the video
Kaggle par 2 cores hote hain — concurrency=2 theek hai. ~3 minute.


In [ ]:
!cd {WORK} && npx remotion render src/index.ts Explainer out/video.mp4 --concurrency=2 2>&1 | tail -5
!ls -lh {WORK}/out/video.mp4


## 7. Watch it


In [ ]:
from IPython.display import Video, display
display(Video(f'{WORK}/out/video.mp4', embed=True, width=960))


## 8. Frames + contact sheet

Yahan verify karo: screen par sab English (captions, keywords, step names, chapters), aur progress bar me parts.


In [ ]:
import subprocess
for f in [45, 250, 700, 1150, 1600, 2350]:
    subprocess.run(['npx','remotion','still','src/index.ts','Explainer',f'out/f{f}.png',f'--frame={f}'], cwd=WORK, check=False, capture_output=True)

cmd = ('ffmpeg -v error -y -i out/f45.png -i out/f250.png -i out/f700.png -i out/f1150.png '
       '-i out/f1600.png -i out/f2350.png -filter_complex "[0][1][2]hstack=3[a];[3][4][5]hstack=3[b];[a][b]vstack=2" sheet.png')
subprocess.run(cmd, cwd=WORK, shell=True, check=False)
from IPython.display import Image, display
display(Image(f'{WORK}/sheet.png'))


## 9. Voice check
Aawaz sun lo — Hindi neural voice honi chahiye.


In [ ]:
from IPython.display import Audio, display
display(Audio(f'{WORK}/public/voice/rag-demo.wav'))


## 10. Apna topic banane ke liye

Sirf **ek file** badalni hai: `src/content.ts`. Uske baad voice + render dobara.

```python
!cd {WORK} && python3 scripts/build_voice.py && npx remotion render src/index.ts Explainer out/video.mp4 --concurrency=2
```

Rules:

| Field | Bhasha | Example |
|---|---|---|
| `say` | **Hindi, Devanagari** + English tech words | `देखो भाई, एक language model अपनी याद से जवाब देता है` |
| `subtitle` | **English only**, 1 chhoti line | `The model answers from memory` |
| `keyword` | **English only**, bada word | `Wrong answer` |
| `steps[].label` | **English only** | `CHUNK`, `EMBED` |

Scene types: `hook` · `concept` · `steps` · `compare` · `number` · `recap`

Offline voice (agar edge-tts block ho):
```python
!cd {WORK} && pip install -q piper-tts && python3 scripts/build_voice.py --engine piper
```


## Troubleshooting

| Problem | Fix |
|---|---|
| clone/npm fail | Notebook settings → Internet **On** |
| edge-tts error / empty audio | `--engine piper` (offline) ya dobara chalao |
| Chrome lib error | cell 1 dobara chalao |
| render slow | concurrency=2 hi rakho (2 cores) |
| Hindi aawaz galat | check karo `say` **Devanagari** me hai (Latin Hindi galat bolti hai) |
| disk full | `rm -rf /kaggle/working/kathareel/voice_cache` |


## 6 · Web studio (recommended)

Terminal ki zaroorat nahi — browser me script edit karo aur video banao.

Kaggle me **Internet ON** rakho, phir neeche wala cell chalao. Banner me `PUBLIC` URL
aayega (ya `https://www.kaggle.com/code/<user>/<notebook>/proxy/8000/`) — usse kholo.

Page par: topic + minutes → script banti hai → har line edit karo → **Generate video** →
video usi page par play hogi.

In [ ]:
!python web/kaggle_launch.py